# Virtual Datasets: From NetCDF to JSON References

This notebook demonstrates how to create a **virtual dataset** from a single remote NetCDF file and save it as a lightweight JSON reference file.

The JSON file contains only *metadata and byte-range pointers* — no data is copied. You can share or commit it, then open it with xarray just like a regular dataset.

**Steps:**
1. Open a single NetCDF file from S3 as a virtual dataset using `virtualizarr`
2. Inspect the virtual dataset (what's in it, how big is the JSON?)
3. Write the references to a local JSON file
4. Re-open with xarray via `fsspec`'s `ReferenceFileSystem`

## Setup

In [1]:
import json
import os
import warnings

import fsspec
import xarray as xr
from dotenv import load_dotenv
from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import S3Store
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from zarr.errors import ZarrUserWarning

warnings.filterwarnings("ignore", category=ZarrUserWarning)

_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env', override=True)

storage_endpoint = os.environ['ENDPOINT_URL']
storage_bucket = 'protocoast-data'
bucket_url = f"s3://{storage_bucket}"

## Step 1: Open one NetCDF file as a virtual dataset

We use `open_virtual_dataset` to read only the file's metadata (headers, chunk layout). No data arrays are downloaded.

`loadable_variables=["time"]` tells virtualizarr to actually load the `time` coordinate values into memory — necessary here because the time axis is needed for concatenation later. Everything else stays as a virtual pointer.

In [2]:
# Pick one forecast file to demonstrate with
nc_url = f"s3://{storage_bucket}/full_dataset/shyfem/taranto/forecast/20250909/taranto_nos_20250909_nc4.nc"

# Build an ObjectStore registry so virtualizarr can read from this S3-compatible endpoint
store_obj = S3Store(
    bucket=storage_bucket,
    endpoint=storage_endpoint,
    region="not-used",
)
registry = ObjectStoreRegistry({bucket_url: store_obj})
parser = HDFParser()

# Open as virtual dataset — reads only metadata, not data arrays
vds = open_virtual_dataset(nc_url, parser=parser, registry=registry, loadable_variables=["time"])
vds

<xarray.Dataset> Size: 2GB
Dimensions:        (time: 144, element: 58285, vertex: 3, node: 30731, level: 70)
Coordinates:
  * time           (time) datetime64[ns] 1kB 2025-09-08 ... 2025-09-13T23:00:00
    level          (level) float32 280B ManifestArray<shape=(70,), dtype=floa...
Dimensions without coordinates: element, vertex, node
Data variables:
    element_index  (element, vertex) int32 699kB ManifestArray<shape=(58285, ...
    latitude       (node) float32 123kB ManifestArray<shape=(30731,), dtype=f...
    longitude      (node) float32 123kB ManifestArray<shape=(30731,), dtype=f...
    salinity       (time, node, level) float32 1GB ManifestArray<shape=(144, ...
    temperature    (time, node, level) float32 1GB ManifestArray<shape=(144, ...
    topology       int32 4B ManifestArray<shape=(), dtype=int32, chunks=()>
    total_depth    (node) float32 123kB ManifestArray<shape=(30731,), dtype=f...
Attributes:
    Conventions:  CF-1.4
    title:        uae
    history:      Wed Sep 10 15:12:42 2025: ncks -4 -L 5 -O -d time,0,143 --c...
    institution:  Centro Euro-Mediterraneo sui Cambiamenti Climatici - CMCC, ...
    source:       Model data produced by SHYFEM-MPI at CMCC
    references:   https://zenodo.org/record/5596734#.Y-YwpxPMLx8
    NCO:          netCDF Operators version 5.0.6 (Homepage = http://nco.sf.ne...

## Step 2: Inspect the virtual dataset

Variables show as `ManifestArray` objects — a list of byte-range references, not actual data. You can inspect the manifest for any variable to see exactly what chunks point where.

In [3]:
# Show the manifest for salinity: path, byte offset, length for each chunk
manifest = vds['salinity'].data.manifest
for chunk_idx, entry in list(manifest.dict().items())[:3]:
    print(f"  chunk {chunk_idx}: {entry}")

  chunk 0.0.0: {'path': 's3://protocoast-data/full_dataset/shyfem/taranto/forecast/20250909/taranto_nos_20250909_nc4.nc', 'offset': 30838, 'length': 2125587}
  chunk 0.0.1: {'path': 's3://protocoast-data/full_dataset/shyfem/taranto/forecast/20250909/taranto_nos_20250909_nc4.nc', 'offset': 2156425, 'length': 2109673}
  chunk 0.0.2: {'path': 's3://protocoast-data/full_dataset/shyfem/taranto/forecast/20250909/taranto_nos_20250909_nc4.nc', 'offset': 4266098, 'length': 2076736}


## Step 3: Write references to a local JSON file

`to_kerchunk(filepath, format='json')` serializes the entire virtual dataset as a [kerchunk reference spec](https://fsspec.github.io/kerchunk/spec.html) JSON file. This file is typically a few KB — tiny compared to the original NetCDF.

In [4]:
ref_path = "taranto_nos_20250909.json"

vds.virtualize.to_kerchunk(ref_path, format="json")

# Check file size
size_kb = os.path.getsize(ref_path) / 1024
print(f"Written: {ref_path}  ({size_kb:.1f} KB)")

Written: taranto_nos_20250909.json  (94.9 KB)


In [5]:
# Peek at the structure of the JSON
with open(ref_path) as f:
    refs = json.load(f)

print("Top-level keys:", list(refs.keys()))
print("Number of chunk references:", len(refs["refs"]))

Top-level keys: ['version', 'refs']
Number of chunk references: 659


## Step 4: Open the JSON references with xarray

`fsspec`'s `ReferenceFileSystem` reads the JSON and intercepts I/O calls — when xarray requests a chunk, fsspec issues a ranged HTTP/S3 request to the original NetCDF file and returns just that slice of bytes.

In [6]:
fs_ref = fsspec.filesystem(
    "reference",
    fo=ref_path,
    remote_protocol="s3",
    remote_options={
        "anon": False,
        "endpoint_url": storage_endpoint,
        "asynchronous": True,
    },
    asynchronous=True,              # both ref-fs and remote s3fs must agree
    skip_instance_cache=True,
)

ds = xr.open_dataset(
    fs_ref.get_mapper(""),
    engine="zarr",
    consolidated=False,
    chunks={},
)
ds

<xarray.Dataset> Size: 2GB
Dimensions:        (element: 58285, vertex: 3, node: 30731, time: 144, level: 70)
Coordinates:
  * time           (time) datetime64[ns] 1kB 2025-09-08 ... 2025-09-13T23:00:00
  * level          (level) float32 280B 2.0 4.0 6.0 ... 1.5e+03 2e+03 2.5e+03
Dimensions without coordinates: element, vertex, node
Data variables:
    element_index  (element, vertex) float64 1MB dask.array<chunksize=(58285, 3), meta=np.ndarray>
    latitude       (node) float32 123kB dask.array<chunksize=(16000,), meta=np.ndarray>
    longitude      (node) float32 123kB dask.array<chunksize=(16000,), meta=np.ndarray>
    salinity       (time, node, level) float32 1GB dask.array<chunksize=(72, 16000, 1), meta=np.ndarray>
    temperature    (time, node, level) float32 1GB dask.array<chunksize=(72, 16000, 1), meta=np.ndarray>
    topology       float64 8B ...
    total_depth    (node) float32 123kB dask.array<chunksize=(16000,), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.4
    title:        uae
    history:      Wed Sep 10 15:12:42 2025: ncks -4 -L 5 -O -d time,0,143 --c...
    institution:  Centro Euro-Mediterraneo sui Cambiamenti Climatici - CMCC, ...
    source:       Model data produced by SHYFEM-MPI at CMCC
    references:   https://zenodo.org/record/5596734#.Y-YwpxPMLx8
    NCO:          netCDF Operators version 5.0.6 (Homepage = http://nco.sf.ne...

## Step 5: Load a small slice to verify data access

Everything so far has been metadata-only. This cell triggers real S3 I/O — only the requested chunk bytes are downloaded.

In [7]:
# Load just the first time step of salinity — triggers real S3 byte-range fetch
sample = ds['salinity'].isel(time=0).compute()
print(sample)

<xarray.DataArray 'salinity' (node: 30731, level: 70)> Size: 9MB
array([[38.831833, 38.831833, 38.829998, ...,  0.      ,  0.      ,
         0.      ],
       [38.831806, 38.831806, 38.82997 , ...,  0.      ,  0.      ,
         0.      ],
       [38.83184 , 38.83184 , 38.830006, ...,  0.      ,  0.      ,
         0.      ],
       ...,
       [38.65799 , 38.65799 , 38.656017, ...,  0.      ,  0.      ,
         0.      ],
       [38.65634 , 38.65634 , 38.65404 , ...,  0.      ,  0.      ,
         0.      ],
       [38.657364, 38.657364, 38.655247, ...,  0.      ,  0.      ,
         0.      ]], shape=(30731, 70), dtype=float32)
Coordinates:
    time     datetime64[ns] 8B 2025-09-08
  * level    (level) float32 280B 2.0 4.0 6.0 8.0 ... 1.5e+03 2e+03 2.5e+03
Dimensions without coordinates: node
Attributes:
    units:          1e-3
    standard_name:  sea_water_salinity
    valid_min:      0.0
    valid_max:      200.0


## Summary

| Step | What happened |
|------|--------------|
| `open_virtual_dataset` | Read only the NetCDF file headers — no data downloaded |
| `to_kerchunk(..., format='json')` | Wrote a small JSON with chunk paths + byte ranges |
| `fsspec.filesystem("reference", ...)` | Mapped the JSON into a Zarr-compatible store |
| `xr.open_dataset(..., engine="zarr")` | Opened lazily — still no data downloaded |
| `.compute()` | Triggered the first real data fetch, one chunk at a time |

The JSON file can be committed to git, shared, or used as a stepping stone toward an [Icechunk](https://icechunk.io) store for mutability and version history.